<a href="https://colab.research.google.com/github/dudanogueira/teofiloAI/blob/main/curso_agente_leis_teofilo_otoni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 🏛️ Construindo um Agente de IA para a Legislação de Teófilo Otoni

### Mini curso prático: Python + LangChain + Weaviate · 2 horas

Ao final desta aula você terá um **agente de IA que responde perguntas sobre as leis do município de Teófilo Otoni/MG**, citando a fonte exata de cada resposta.

Vamos usar dados **reais e públicos** do portal da Câmara Municipal.

---

## O que vamos construir

```
   Portal da Câmara                 Este notebook                    Você pergunta
  ┌────────────────┐            ┌────────────────────┐            ┌───────────────┐
  │ API de leis    │ ──────►    │ 1. Coleta          │            │ "Quantos      │
  │ (JSON)         │            │ 2. Extração de PDF │            │  vereadores?" │
  │                │            │ 3. Chunking        │            └───────┬───────┘
  │ PDFs das leis  │ ──────►    │ 4. Vetorização     │                    │
  │                │            └─────────┬──────────┘                    ▼
  │ Lei Orgânica   │ ──────►              │                    ┌──────────────────┐
  └────────────────┘                      ▼                    │   AGENTE (LLM)   │
                              ┌───────────────────────┐        │                  │
                              │      WEAVIATE         │        │  Decide sozinho  │
                              │  ┌─────────────────┐  │◄───────┤  qual base usar  │
                              │  │ tenant: leis    │  │        │                  │
                              │  ├─────────────────┤  │        └────────┬─────────┘
                              │  │ tenant:         │  │                 │
                              │  │  lei_organica   │  │                 ▼
                              │  └─────────────────┘  │        Resposta com a
                              └───────────────────────┘        fonte citada
```

O ponto central da aula: **o agente tem duas bases de conhecimento separadas** e precisa decidir sozinho qual consultar.

| Base | O que é | Quando usar |
|------|---------|-------------|
| **Lei Orgânica** | A "constituição" do município. Define estrutura de poderes, competências, mandatos. | "Quantos vereadores tem a Câmara?" |
| **Leis municipais** | Leis ordinárias do dia a dia (2024–2026). Regras pontuais e concretas. | "Existe lei sobre motoristas de aplicativo?" |

---

## Agenda

| Tempo | Parte | Assunto |
|------:|:-----:|---------|
| 10 min | **0** | Preparação e chaves de acesso |
| 20 min | **1** | Python na prática: consumindo a API da Câmara |
| 20 min | **2** | De PDF para texto: 194 leis e a Lei Orgânica |
| 15 min | **3** | Chunking com LangChain |
| 30 min | **4** | Weaviate: busca vetorial e multi-tenancy |
| 30 min | **5** | O agente: LangChain + Gemini + 2 ferramentas |
|  5 min | **6** | Encerramento e próximos passos |

> 💡 **Como usar este notebook:** execute as células **na ordem**, de cima para baixo (`Shift + Enter`). Células marcadas com 🎯 são exercícios para você tentar sozinho.

---
# Parte 0 — Preparação (10 min)

## 0.1 Instalando as bibliotecas

O Colab já vem com Python e muita coisa instalada, mas precisamos de quatro pacotes:

| Pacote | Para quê |
|--------|----------|
| `weaviate-client` | Falar com o banco de dados vetorial |
| `langchain` + `langgraph` | Montar o agente e as ferramentas |
| `langchain-google-genai` | Conectar o Gemini ao LangChain |
| `pypdf` | Extrair texto dos PDFs das leis |

As versões estão **fixadas** de propósito: garante que a aula funcione igual para todo mundo.

In [ ]:
%pip install -q \
    "weaviate-client==4.23.1" \
    "langchain==1.4.0" \
    "langchain-google-genai==4.4.0" \
    "langgraph==1.2.11" \
    "pypdf==6.18.1" \
    "requests"

print("✅ Bibliotecas instaladas!")

## 0.2 As duas chaves que você precisa

Este notebook usa dois serviços externos:

### 🔵 Weaviate Cloud — o banco de dados vetorial
Onde as leis ficam armazenadas e pesquisáveis. **Não precisa de chave de embeddings**: o Weaviate Cloud já inclui o serviço de vetorização (Weaviate Embeddings), então não há custo adicional nem outra conta para criar.

- **Cada aluno cria o seu próprio cluster grátis** em **[console.weaviate.cloud](https://console.weaviate.cloud)** (leva ~2 minutos, sem cartão). O porquê está logo abaixo, na seção 0.3.

### 🟡 Google Gemini — o cérebro do agente
É o modelo de linguagem que lê os resultados da busca e escreve a resposta.

- Pegue sua chave grátis em **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**.

---

### 🔐 Guardando as chaves com segurança

**Nunca escreva uma chave direto no código do notebook.** Se você compartilhar o arquivo, a chave vai junto.

O Colab tem um cofre para isso: o ícone de **🔑 chave** na barra lateral esquerda. Crie três segredos e **ative o botão "Acesso ao notebook"** em cada um:

| Nome do segredo | Valor |
|-----------------|-------|
| `WEAVIATE_URL`  | o endereço do seu cluster (sem o `https://`) |
| `WEAVIATE_API_KEY` | a chave de API do cluster |
| `GEMINI_API_KEY` | a chave do Google AI Studio |

A célula abaixo lê o cofre. Se você não estiver no Colab (ou não tiver criado os segredos), ela pergunta as chaves na hora.

In [ ]:
import os
from getpass import getpass


def pegar_segredo(nome: str, descricao: str) -> str:
    """Le um segredo do cofre do Colab; se nao existir, pergunta ao usuario."""
    try:
        from google.colab import userdata
        valor = userdata.get(nome)
        if valor:
            print(f"🔑 {nome}: lido do cofre do Colab")
            return valor.strip()
    except Exception:
        pass

    valor = os.environ.get(nome)
    if valor:
        print(f"🔑 {nome}: lido das variáveis de ambiente")
        return valor.strip()

    return getpass(f"Cole aqui {descricao}: ").strip()


WEAVIATE_URL = pegar_segredo("WEAVIATE_URL", "a URL do cluster Weaviate")
WEAVIATE_API_KEY = pegar_segredo("WEAVIATE_API_KEY", "a API key do Weaviate")
GEMINI_API_KEY = pegar_segredo("GEMINI_API_KEY", "a API key do Google Gemini")

# Limpa erros comuns de copiar e colar
WEAVIATE_URL = WEAVIATE_URL.replace("https://", "").replace("http://", "").rstrip("/")

print(f"\n📍 Cluster: {WEAVIATE_URL}")

## 0.3 ⚠️ Antes de seguir: os limites do plano gratuito

O plano **Free** do Weaviate Cloud é excelente para aprender, mas tem dois limites que afetam diretamente esta aula:

| Limite do plano Free | Valor |
|---|---|
| Coleções por cluster | **1** |
| Tenants por cluster | **3** |

Por isso, **nesta aula cada participante usa o seu próprio cluster**. Não é um contorno: é o jeito certo de fazer. Você sai daqui com o ambiente montado na sua conta, podendo continuar mexendo depois da aula.

> 🧑‍🎓 **Ainda não criou?** Vá em [console.weaviate.cloud](https://console.weaviate.cloud), clique em *Create cluster* → *Sandbox/Free*, espere ~2 minutos e copie a **REST Endpoint URL** e a **API Key**. Não pede cartão de crédito. Depois volte à célula 0.2.

Nosso consumo cabe folgado no plano Free: **1 coleção e 2 tenants**.

E repare: esse limite de **1 coleção** é exatamente o que torna a decisão da Parte 4 interessante. Precisamos de duas bases de conhecimento separadas, mas só podemos criar uma coleção — então a separação vai ter que vir de **multi-tenancy**. Restrição virando arquitetura.

## 0.4 Testando a conexão com o Weaviate

Momento da verdade. Se esta célula falhar, é problema de credencial — resolva antes de seguir.

In [ ]:
import weaviate
from weaviate.classes.init import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
)

print("🟢 Conectado!" if client.is_ready() else "🔴 Não respondeu")
print("Versão do Weaviate:", client.get_meta()["version"])

> ⚠️ **Sobre `client.close()`**
>
> O cliente do Weaviate abre conexões de rede que precisam ser fechadas no final. Vamos fazer isso na última célula do notebook. Se você reiniciar o ambiente do Colab no meio da aula, basta rodar a célula acima de novo.

---
# Parte 1 — Python na prática (20 min)

Em vez de exercícios artificiais, vamos aprender Python **buscando dados de verdade** do portal da Câmara de Teófilo Otoni.

## 1.1 O básico: variáveis, tipos e f-strings

Python descobre o tipo sozinho — você não declara.

In [ ]:
# Texto (str), número inteiro (int), decimal (float) e booleano (bool)
municipio = "Teófilo Otoni"
estado = "MG"
total_vereadores = 19
populacao_mil = 134.4
tem_lei_organica = True

# f-string: coloca o valor de uma variável dentro do texto usando { }
print(f"{municipio}/{estado} tem {total_vereadores} vereadores.")
print(f"População: {populacao_mil:.1f} mil habitantes.")

# type() mostra o tipo de qualquer coisa
for valor in [municipio, total_vereadores, populacao_mil, tem_lei_organica]:
    print(f"  {str(valor):<15} -> {type(valor).__name__}")

## 1.2 Listas e dicionários

Estas são as duas estruturas que você mais vai usar:

- **lista** `[ ]` → uma sequência ordenada de coisas
- **dicionário** `{ }` → pares de `chave: valor`, como uma ficha cadastral

Uma lei do portal chega exatamente no formato de um dicionário.

In [ ]:
# Um dicionário: cada lei é uma "ficha" com campos nomeados
lei = {
    "numero": "7981",
    "ano": "2026",
    "titulo": "Lei",
    "ementa": "Dispõe sobre a instalação de pontos de apoio para trabalhadores de aplicativos.",
}

# Acessando um campo pelo nome da chave
print("Número:", lei["numero"])
print("Ementa:", lei["ementa"])

# .get() é mais seguro: não quebra se a chave não existir
print("Autor: ", lei.get("autor", "(não informado)"))

# Uma lista de dicionários é como os dados vão chegar da API
leis_exemplo = [lei, {"numero": "7980", "ano": "2026", "titulo": "Lei", "ementa": "Subvenção social."}]
print(f"\nTemos {len(leis_exemplo)} leis na lista.")
print("A primeira é a de número", leis_exemplo[0]["numero"])

## 1.3 Buscando dados reais: a API da Câmara

O portal da Câmara ([teofilootoni.mg.leg.br](https://www.teofilootoni.mg.leg.br/legislacoes/todas/Todas%20as%20Leis)) mostra as leis numa tabela. Essa tabela é alimentada por um **endpoint JSON** — um endereço que devolve dados estruturados em vez de página HTML.

Isso é ouro: em vez de raspar HTML (frágil), consumimos os dados direto da fonte.

A biblioteca `requests` faz a chamada HTTP; `.json()` converte a resposta em listas e dicionários do Python.

In [ ]:
import requests

BASE = "https://www.teofilootoni.mg.leg.br"
TIPO_LEI = 4  # no portal: 4 = Lei ordinária, 5 = Lei Complementar, 7 = Decreto...

resposta = requests.get(
    f"{BASE}/legislacoes/get/{TIPO_LEI}",
    params={"draw": 1, "start": 0, "length": 5, "ano": 2026},
    headers={"User-Agent": "Mozilla/5.0", "X-Requested-With": "XMLHttpRequest"},
    timeout=60,
)
resposta.raise_for_status()   # levanta erro se o servidor respondeu 4xx/5xx

dados = resposta.json()
print("Status HTTP:", resposta.status_code)
print("Total de leis ordinárias no portal:", dados["recordsTotal"])
print("Recebemos nesta chamada:", len(dados["data"]))

Vamos olhar **um** registro por dentro. `json.dumps(..., indent=2)` imprime de forma legível.

In [ ]:
import json

primeira = dados["data"][0]
print(json.dumps(primeira, indent=2, ensure_ascii=False)[:1200])

Repare nos campos que interessam:

- `numero`, `ano`, `titulo` → identificam a lei
- `ementa` → o resumo oficial, escrito por humanos. **Ótimo material para busca semântica.**
- `arquivo.path` → o caminho do PDF com o texto integral
- `id` → serve para montar o link da página da lei

## 1.4 Transformando em função

Copiar e colar o mesmo código mudando o ano é receita para bug. **Função** = um bloco de código com nome, que recebe parâmetros e devolve um resultado.

In [ ]:
def listar_leis(ano: int, tipo: int = TIPO_LEI, limite: int = 2000) -> list[dict]:
    """Devolve a lista de leis de um determinado ano no portal da Câmara.

    Args:
        ano: o ano das leis (ex: 2025).
        tipo: código do tipo de norma no portal (4 = Lei ordinária).
        limite: máximo de registros a trazer.
    """
    resposta = requests.get(
        f"{BASE}/legislacoes/get/{tipo}",
        params={"draw": 1, "start": 0, "length": limite, "ano": ano},
        headers={"User-Agent": "Mozilla/5.0", "X-Requested-With": "XMLHttpRequest"},
        timeout=60,
    )
    resposta.raise_for_status()
    return resposta.json()["data"]


# Testando
leis_2026 = listar_leis(2026)
print(f"Leis de 2026: {len(leis_2026)}")
print("Mais recente:", leis_2026[0]["numero"], "-", leis_2026[0]["ementa"][:70])

## 1.5 Laços e list comprehensions

Um **`for`** percorre uma lista. A **list comprehension** faz a mesma coisa em uma linha — é o jeito mais idiomático em Python.

In [ ]:
# Jeito tradicional
ementas = []
for lei in leis_2026[:5]:
    ementas.append(lei["ementa"])

# Jeito Python: list comprehension  ->  [ o_que_quero  for item in lista ]
ementas = [lei["ementa"] for lei in leis_2026[:5]]

for i, ementa in enumerate(ementas, start=1):
    print(f"{i}. {ementa[:85]}")

A list comprehension também **filtra**, com um `if` no final:

In [ ]:
# Só as leis cuja ementa menciona "saúde"
sobre_saude = [lei for lei in leis_2026 if "saúde" in lei["ementa"].lower()]

print(f"{len(sobre_saude)} de {len(leis_2026)} leis de 2026 mencionam 'saúde':\n")
for lei in sobre_saude:
    print(f"  Lei {lei['numero']}/{lei['ano']}: {lei['ementa'][:80]}")

> 🔍 **Guarde esta observação** — ela é o motivo de existir o resto da aula.
>
> O filtro acima é **busca por palavra exata**. Ele encontra "saúde", mas **não** encontraria uma lei que fala de "atendimento médico", "UPA" ou "hospital" sem usar a palavra "saúde".
>
> É exatamente esse problema que a **busca vetorial** resolve — e é o que vamos montar na Parte 4.

## 🎯 Exercício 1

Escreva uma função `contar_por_ano(anos)` que recebe uma lista de anos e imprime quantas leis existem em cada um.

<details>
<summary>👀 Ver solução</summary>

```python
def contar_por_ano(anos: list[int]) -> dict[int, int]:
    contagem = {}
    for ano in anos:
        contagem[ano] = len(listar_leis(ano))
        print(f"  {ano}: {contagem[ano]} leis")
    return contagem

contar_por_ano([2022, 2023, 2024, 2025, 2026])
```
</details>

In [ ]:
# 🎯 Sua vez! Escreva a função aqui embaixo.

---
# Parte 2 — De PDF para texto (20 min)

A ementa é um bom resumo, mas o agente precisa do **texto integral** para responder com precisão. E o texto integral está em PDF.

Nesta parte vamos:
1. Descobrir a URL do PDF de cada lei
2. Baixar e extrair o texto
3. Fazer isso para ~200 leis **em paralelo**
4. Baixar e estruturar a Lei Orgânica

## 2.1 Montando a URL do PDF

No JSON, o campo `arquivo.path` traz o caminho do arquivo. Basta juntar com o endereço do storage. Como o caminho tem espaços e acentos, usamos `urllib.parse.quote` para codificá-lo corretamente.

In [ ]:
import urllib.parse

S3 = "https://digitaliza-institucional.s3.us-east-2.amazonaws.com/"


def url_do_pdf(lei: dict) -> str | None:
    """Monta a URL publica do PDF de uma lei, ou None se nao houver arquivo."""
    arquivo = lei.get("arquivo") or {}
    caminho = arquivo.get("path")
    return S3 + urllib.parse.quote(caminho) if caminho else None


exemplo = leis_2026[0]
print("Lei", exemplo["numero"], "/", exemplo["ano"])
print(url_do_pdf(exemplo))

## 2.2 Extraindo o texto de um PDF

`requests.get(...).content` traz os **bytes** do arquivo. O `pypdf` lê esses bytes e extrai o texto página por página.

`io.BytesIO` é o truque que permite tratar bytes na memória como se fossem um arquivo — assim não precisamos salvar nada em disco.

In [ ]:
import io
from pypdf import PdfReader


def baixar_pdf(url: str) -> bytes:
    """Baixa um PDF e devolve os bytes."""
    resposta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
    resposta.raise_for_status()
    return resposta.content


def pdf_para_texto(conteudo: bytes) -> str:
    """Extrai o texto de todas as paginas de um PDF."""
    leitor = PdfReader(io.BytesIO(conteudo))
    return "\n".join((pagina.extract_text() or "") for pagina in leitor.pages)


bytes_pdf = baixar_pdf(url_do_pdf(exemplo))
texto = pdf_para_texto(bytes_pdf)

print(f"📄 {len(bytes_pdf) // 1024} KB · {len(texto)} caracteres extraídos\n")
print(texto[:700])

> 💡 **Nem todo PDF coopera.** Este portal usa PDFs com "camada de texto" — o texto está lá, selecionável. Se fossem imagens escaneadas, `extract_text()` devolveria vazio e precisaríamos de **OCR** (ex: `pytesseract`). Sempre confira o resultado antes de confiar.

## 2.3 Limpando o texto

A extração traz ruído: espaços duplicados, quebras de linha sobrando. Vamos limpar com **expressões regulares** (`re`), que buscam padrões em texto.

In [ ]:
import html
import re


def limpar(texto: str) -> str:
    """Normaliza espacos, quebras de linha e entidades HTML."""
    texto = html.unescape(texto)               # &quot; -> "  ·  &amp; -> &
    texto = re.sub(r"[ \t]+", " ", texto)      # vários espaços -> um só
    texto = re.sub(r"\n{3,}", "\n\n", texto)   # 3+ quebras -> 2
    return texto.strip()


print(limpar(texto)[:400])

## 2.4 Baixando ~200 leis em paralelo

Baixar 200 PDFs **um por vez** levaria vários minutos, pois a maior parte do tempo é só espera de rede.

O `ThreadPoolExecutor` resolve: ele mantém várias tarefas "esperando" ao mesmo tempo. O resultado cai de minutos para **menos de um minuto**.

Repare também no `try/except`: com 200 downloads, **algum vai falhar**. Código que lida com o mundo real precisa tolerar falhas sem derrubar tudo.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

ANOS = [2024, 2025, 2026]  # 👈 quer uma base maior? acrescente anos aqui


def processar_lei(lei: dict) -> dict | None:
    """Baixa o PDF de uma lei e devolve a lei com o texto extraido. None se falhar."""
    try:
        texto = limpar(pdf_para_texto(baixar_pdf(url_do_pdf(lei))))
        if len(texto) < 200:          # PDF vazio ou só imagem
            return None
        return {**lei, "texto_extraido": texto}
    except Exception as erro:
        print(f"  ⚠️ falhou {lei.get('numero')}/{lei.get('ano')}: {type(erro).__name__}")
        return None


# 1) Lista todas as leis dos anos escolhidos que tenham PDF
catalogo = []
for ano in ANOS:
    catalogo += [lei for lei in listar_leis(ano) if url_do_pdf(lei)]
print(f"📚 {len(catalogo)} leis no catálogo. Baixando os PDFs...\n")

# 2) Baixa e extrai em paralelo
with ThreadPoolExecutor(max_workers=8) as executor:
    leis = [resultado for resultado in executor.map(processar_lei, catalogo) if resultado]

total_chars = sum(len(lei["texto_extraido"]) for lei in leis)
print(f"\n✅ {len(leis)} leis com texto extraído · {total_chars:,} caracteres no total")

In [ ]:
# Como ficou a distribuição de tamanho? (dados reais nunca são uniformes)
tamanhos = sorted((len(lei["texto_extraido"]), lei["numero"], lei["ano"]) for lei in leis)

print("As 3 menores:")
for n, numero, ano in tamanhos[:3]:
    print(f"  Lei {numero}/{ano}: {n:>7,} caracteres")
print("\nAs 3 maiores:")
for n, numero, ano in tamanhos[-3:]:
    print(f"  Lei {numero}/{ano}: {n:>7,} caracteres")
print(f"\nMediana: {tamanhos[len(tamanhos) // 2][0]:,} caracteres")

## 2.5 A Lei Orgânica: um documento, muita estrutura

A Lei Orgânica é diferente das leis ordinárias:

- É **um único PDF de 112 páginas**
- É **hierarquicamente organizada**: `TÍTULO` → `CAPÍTULO` → `Seção` → `Art. N`

Essa estrutura é um presente. Em vez de picotar o texto em pedaços arbitrários, podemos cortar **por artigo** — que é justamente a unidade que um jurista cita: *"o Art. 21 da Lei Orgânica"*.

In [ ]:
LEI_ORGANICA_URL = S3 + "teofilo-otoni-camara-municipal/site/lei-organica-completa-com-emendas.pdf"

leitor_lo = PdfReader(io.BytesIO(baixar_pdf(LEI_ORGANICA_URL)))
paginas_lo = [(numero, pagina.extract_text() or "") for numero, pagina in enumerate(leitor_lo.pages, start=1)]

print(f"📕 Lei Orgânica: {len(paginas_lo)} páginas")
print("\n--- Início do documento ---")
print(paginas_lo[0][1][:450])

### Achando os marcadores com regex

Antes de escrever o parser, vamos **conferir** que os padrões realmente aparecem. Nunca confie na estrutura de um documento sem olhar.

In [ ]:
texto_lo = "\n".join(t for _, t in paginas_lo)

padroes = {
    "TÍTULO":   r"T[ÍI]TULO\s+[IVXL]+",
    "CAPÍTULO": r"CAP[ÍI]TULO\s+[IVXL]+",
    "Artigo":   r"\bArt\s*\.\s*\d+",
}

for nome, padrao in padroes.items():
    achados = re.findall(padrao, texto_lo)
    print(f"{nome:<10} {len(achados):>4} ocorrências   ex: {achados[:4]}")

### O parser: percorrendo o documento linha a linha

A lógica é simples de descrever:

> Leia linha por linha. Quando encontrar `TÍTULO`/`CAPÍTULO`/`Seção`, **anote onde você está**. Quando encontrar `Art. N`, **comece a juntar** um artigo novo. Qualquer outra linha, **acumule** no artigo atual.

Ao final, cada artigo sai com o texto **e** o endereço dele dentro da lei.

In [ ]:
RE_ARTIGO   = re.compile(r"^\s*Art\s*\.?\s*(\d+)\s*[-–ºo°\.\s]", re.IGNORECASE)
RE_TITULO   = re.compile(r"^\s*T[ÍI]TULO\s+([IVXL]+)\s*$")
RE_CAPITULO = re.compile(r"^\s*CAP[ÍI]TULO\s+([IVXL]+)\s*$")
RE_SECAO    = re.compile(r"^\s*Se[çc][ãa]o\s+([IVXL]+)\s*$", re.IGNORECASE)
RE_NUM_PAG  = re.compile(r"^\s*\d{1,3}\s*$")   # número de página solto


def parse_lei_organica(paginas: list[tuple[int, str]]) -> list[dict]:
    """Quebra a Lei Organica em artigos, guardando TITULO/CAPITULO/Secao de cada um."""
    # Achata tudo em (linha, numero_da_pagina), descartando lixo
    linhas = []
    for num_pagina, texto_pagina in paginas:
        for linha in texto_pagina.split("\n"):
            linha = re.sub(r"[ \t]+", " ", linha).rstrip()
            if linha.strip() and not RE_NUM_PAG.match(linha):
                linhas.append((linha, num_pagina))

    # O documento começa com um sumário; o corpo começa no primeiro "TÍTULO I" sozinho
    inicio = next(i for i, (linha, _) in enumerate(linhas) if RE_TITULO.match(linha))

    titulo = capitulo = secao = ""
    artigos, atual = [], None

    def fechar_artigo():
        if atual and atual["corpo"]:
            atual["texto"] = limpar("\n".join(atual.pop("corpo")))
            artigos.append(atual)

    i = inicio
    while i < len(linhas):
        linha, pagina = linhas[i]
        proxima = linhas[i + 1][0].strip() if i + 1 < len(linhas) else ""
        # o nome do TÍTULO/CAPÍTULO vem na linha seguinte, em maiúsculas
        tem_nome = bool(proxima) and proxima.isupper()

        if RE_TITULO.match(linha):
            fechar_artigo(); atual = None
            titulo = f"{linha.strip()} - {proxima}" if tem_nome else linha.strip()
            capitulo = secao = ""
            i += 2 if tem_nome else 1
        elif RE_CAPITULO.match(linha):
            fechar_artigo(); atual = None
            capitulo = f"{linha.strip()} - {proxima}" if tem_nome else linha.strip()
            secao = ""
            i += 2 if tem_nome else 1
        elif RE_SECAO.match(linha):
            fechar_artigo(); atual = None
            secao = f"{linha.strip()} - {proxima}" if proxima else linha.strip()
            i += 2 if proxima else 1
        elif (achou := RE_ARTIGO.match(linha)):
            fechar_artigo()
            numero = int(achou.group(1))
            atual = {
                "numero_artigo": numero, "artigo": f"Art. {numero}",
                "titulo": titulo, "capitulo": capitulo, "secao": secao,
                "pagina": pagina, "corpo": [linha.strip()],
            }
            i += 1
        else:
            if atual:
                atual["corpo"].append(linha.strip())
            i += 1

    fechar_artigo()
    return artigos


artigos = parse_lei_organica(paginas_lo)
print(f"📑 {len(artigos)} artigos extraídos da Lei Orgânica")

In [ ]:
# Conferindo: o artigo que define o número de vereadores
for artigo in artigos:
    if artigo["numero_artigo"] == 21:
        print(f"📍 {artigo['artigo']}  (página {artigo['pagina']})")
        print(f"   {artigo['titulo']}")
        print(f"   {artigo['capitulo']}")
        print(f"   {artigo['secao']}\n")
        print(artigo["texto"][:420])
        break

> ✅ **Por que isso importa tanto?**
>
> Cada artigo agora carrega o próprio **endereço** dentro da lei. Quando o agente citar uma resposta, ele vai poder dizer *"Art. 21, TÍTULO IV — DA ORGANIZAÇÃO DOS PODERES MUNICIPAIS, página 16"* em vez de despejar um trecho solto.
>
> **Metadado bom é o que separa um RAG confiável de um chute bem escrito.**

---
# Parte 3 — Chunking com LangChain (15 min)

## 3.1 Por que picotar o texto?

Não dá para jogar 900 mil caracteres de leis dentro do modelo a cada pergunta. Precisamos **recuperar só os trechos relevantes** — e para isso o texto precisa estar dividido em pedaços (*chunks*).

O tamanho do chunk é um **equilíbrio**:

| Chunk muito pequeno | Chunk muito grande |
|---|---|
| Perde o contexto ao redor | Mistura vários assuntos num vetor só |
| "…será de 6 meses." (6 meses do quê?) | A busca fica imprecisa |

Na prática: **800–2000 caracteres**, com uma **sobreposição** (*overlap*) entre chunks vizinhos para não cortar uma frase importante ao meio.

## 3.2 O `RecursiveCharacterTextSplitter`

É o divisor mais usado do LangChain. O nome assusta, a ideia é simples:

> Tente quebrar no primeiro separador da lista. Se o pedaço ainda ficou grande demais, tente o próximo separador. E assim por diante.

Colocando `"\nArt. "` como **primeiro** separador, pedimos ao divisor que **prefira cortar entre artigos** — respeitando a estrutura do documento jurídico.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Demonstração com um texto curto para você ver o mecanismo
demo = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=30,
    separators=["\nArt. ", "\n\n", "\n", ". ", " "],
)

texto_demo = (
    "Art. 1º Fica instituído o programa municipal de incentivo à leitura.\n"
    "Art. 2º O programa será coordenado pela Secretaria de Educação, com apoio "
    "das bibliotecas públicas e escolas da rede municipal de ensino.\n"
    "Art. 3º Esta lei entra em vigor na data de sua publicação."
)

for i, pedaco in enumerate(demo.split_text(texto_demo)):
    print(f"--- chunk {i} ({len(pedaco)} chars) ---")
    print(pedaco, "\n")

Repare que os cortes caíram **entre artigos**, e não no meio de uma frase. É o `separators` fazendo efeito.

## 3.3 Chunking das leis municipais

Agora para valer. Cada chunk vira um objeto que vai para o banco, carregando **junto** os metadados da lei de origem.

In [ ]:
divisor_leis = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\nArt. ", "\nParágrafo", "\n\n", "\n", ". ", " "],
)


def montar_objetos_leis(leis: list[dict]) -> list[dict]:
    """Transforma cada lei em um ou mais objetos prontos para o Weaviate."""
    objetos = []
    for lei in leis:
        pedacos = divisor_leis.split_text(lei["texto_extraido"])
        for indice, pedaco in enumerate(pedacos):
            objetos.append({
                "fonte": "lei_municipal",
                "referencia": f"Lei nº {lei['numero']}/{lei['ano']}",
                "numero": str(lei["numero"]),
                "ano": int(lei["ano"]),
                "ementa": limpar(lei["ementa"] or ""),
                "contexto": "",
                "texto": pedaco,
                "chunk_id": indice,
                "total_chunks": len(pedacos),
                "pagina": 0,
                "url_pdf": url_do_pdf(lei),
                "url_pagina": f"{BASE}/legislacoes/show/{lei['id']}",
            })
    return objetos


objetos_leis = montar_objetos_leis(leis)
print(f"✂️  {len(leis)} leis → {len(objetos_leis)} chunks")
print(f"    média de {len(objetos_leis) / len(leis):.1f} chunks por lei\n")
print(json.dumps(objetos_leis[0], indent=2, ensure_ascii=False)[:800])

## 3.4 Chunking da Lei Orgânica: estrutura ganha de tamanho

Aqui está a diferença importante da aula.

Nas leis municipais deixamos o divisor cortar por tamanho. Na Lei Orgânica, **já temos os artigos** — então o chunk é o artigo. Só usamos o divisor como rede de segurança, para os poucos artigos muito longos.

Repare no campo `contexto`: juntamos `TÍTULO > CAPÍTULO > Seção` num texto só. Ele **também vai ser vetorizado**, o que enriquece o significado do chunk.

In [ ]:
divisor_lo = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=200,
    separators=["\nParágrafo", "\n§", "\n", ". ", " "],
)


def montar_objetos_lei_organica(artigos: list[dict]) -> list[dict]:
    """Transforma cada artigo da Lei Organica em objeto(s) para o Weaviate."""
    objetos = []
    for artigo in artigos:
        contexto = " > ".join(x for x in (artigo["titulo"], artigo["capitulo"], artigo["secao"]) if x)
        pedacos = divisor_lo.split_text(artigo["texto"]) or [artigo["texto"]]
        for indice, pedaco in enumerate(pedacos):
            objetos.append({
                "fonte": "lei_organica",
                "referencia": artigo["artigo"],
                "numero": str(artigo["numero_artigo"]),
                "ano": 0,
                "ementa": "",
                "contexto": contexto,
                "texto": pedaco,
                "chunk_id": indice,
                "total_chunks": len(pedacos),
                "pagina": artigo["pagina"],
                "url_pdf": LEI_ORGANICA_URL,
                "url_pagina": LEI_ORGANICA_URL,
            })
    return objetos


objetos_lo = montar_objetos_lei_organica(artigos)
grandes = sum(1 for a in artigos if len(a["texto"]) > 1800)

print(f"✂️  {len(artigos)} artigos → {len(objetos_lo)} chunks")
print(f"    apenas {grandes} artigos precisaram ser subdivididos\n")
print(json.dumps(objetos_lo[20], indent=2, ensure_ascii=False)[:800])

> 🧠 **A lição de chunking desta aula**
>
> O melhor chunk não é o de tamanho X. É o que **respeita a unidade natural de significado** do documento.
>
> Em leis, essa unidade é o artigo. Num manual, seria a seção. Numa transcrição, o turno de fala. Olhe o documento antes de escolher o `chunk_size`.

---
# Parte 4 — Weaviate: busca vetorial e multi-tenancy (30 min)

## 4.1 O problema que a busca vetorial resolve

Lá na Parte 1 filtramos leis com `"saúde" in ementa`. Funciona — e falha silenciosamente:

| Pergunta do cidadão | Busca por palavra | Busca vetorial |
|---|---|---|
| "atendimento na UPA" | ❌ não acha "saúde" | ✅ acha |
| "motoboy de entrega" | ❌ não acha "aplicativo" | ✅ acha |
| "nome de rua" | ❌ não acha "logradouro" | ✅ acha |

A busca vetorial converte **texto em números** (um *embedding*) de forma que textos com significado parecido fiquem **próximos no espaço**. A busca deixa de ser "quem contém esta palavra" e passa a ser "quem fala sobre isto".

O Weaviate faz três coisas para nós:
1. **Vetoriza** automaticamente cada objeto na hora de gravar
2. **Armazena** vetor + metadados juntos
3. **Busca** por proximidade em milissegundos, mesmo com milhões de objetos

> 💰 Usamos o **Weaviate Embeddings**, incluso no Weaviate Cloud. O modelo é multilíngue e funciona muito bem em português — e **você não precisa de outra chave de API**.

## 4.2 Multi-tenancy: duas bases, um só lugar

Precisamos manter **as leis municipais e a Lei Orgânica separadas**, porque o agente vai escolher entre elas.

Um jeito seria criar duas coleções. Vamos usar um recurso melhor: **multi-tenancy**.

```
        Collection: Legislacao          (um schema só, definido uma vez)
        ┌──────────────────────────────────────────────────┐
        │   tenant "leis"                tenant "lei_organica"     │
        │  ┌────────────────┐           ┌────────────────┐ │
        │  │ shard próprio  │           │ shard próprio  │ │
        │  │ índice vetorial│           │ índice vetorial│ │
        │  │ próprio        │           │ próprio        │ │
        │  └────────────────┘           └────────────────┘ │
        └──────────────────────────────────────────────────┘
                     isolamento real entre tenants
```

Cada tenant tem **shard e índice vetorial próprios**. Uma busca num tenant **nunca** enxerga o outro — o isolamento é garantido pelo banco, não por um filtro que você pode esquecer de aplicar.

É assim que se constrói SaaS multi-cliente: um schema, milhares de clientes, zero risco de um ver o dado do outro.

> ⚠️ **No plano Free**, lembre-se: 1 coleção e no máximo 3 tenants. Usaremos 2. Em planos pagos, o Weaviate suporta **centenas de milhares** de tenants por coleção, com os inativos descarregados automaticamente da memória.

## 4.3 Definindo o schema

O **schema** diz ao Weaviate quais campos existem e o que fazer com cada um.

Duas decisões importantes aqui:

- **`source_properties=["ementa", "contexto", "texto"]`** — só esses três viram vetor. Não faz sentido vetorizar uma URL.
- **`skip_vectorization=True`** — marca os campos que são só metadado (número, página, link). Eles continuam gravados e filtráveis, mas ficam fora do vetor.

In [ ]:
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.classes.tenants import Tenant

COLECAO = "Legislacao"
TENANT_LEIS = "leis"
TENANT_LO = "lei_organica"

if not client.collections.exists(COLECAO):
    client.collections.create(
        COLECAO,
        description="Legislação do município de Teófilo Otoni/MG",

        # 👇 liga o multi-tenancy
        multi_tenancy_config=Configure.multi_tenancy(enabled=True),

        # 👇 quem transforma texto em vetor, e a partir de quais campos
        vector_config=Configure.Vectors.text2vec_weaviate(
            name="default",
            source_properties=["ementa", "contexto", "texto"],
        ),

        properties=[
            # --- identificação (metadados: não entram no vetor) ---
            Property(name="fonte", data_type=DataType.TEXT,
                     tokenization=Tokenization.FIELD, skip_vectorization=True),
            Property(name="referencia", data_type=DataType.TEXT,
                     tokenization=Tokenization.FIELD, skip_vectorization=True),
            Property(name="numero", data_type=DataType.TEXT,
                     tokenization=Tokenization.FIELD, skip_vectorization=True),
            Property(name="ano", data_type=DataType.INT),

            # --- conteúdo (isto vira vetor) ---
            Property(name="ementa", data_type=DataType.TEXT, description="resumo oficial da lei"),
            Property(name="contexto", data_type=DataType.TEXT, description="TÍTULO > CAPÍTULO"),
            Property(name="texto", data_type=DataType.TEXT, description="o trecho em si"),

            # --- rastreabilidade ---
            Property(name="chunk_id", data_type=DataType.INT, skip_vectorization=True),
            Property(name="total_chunks", data_type=DataType.INT, skip_vectorization=True),
            Property(name="pagina", data_type=DataType.INT, skip_vectorization=True),
            Property(name="url_pdf", data_type=DataType.TEXT, skip_vectorization=True),
            Property(name="url_pagina", data_type=DataType.TEXT, skip_vectorization=True),
        ],
    )
    print(f"✅ Coleção '{COLECAO}' criada")
else:
    print(f"ℹ️  Coleção '{COLECAO}' já existe — vamos reaproveitar o schema")

legislacao = client.collections.get(COLECAO)

## 4.4 Criando os seus tenants

Repare que apagamos e recriamos os **tenants**, mas nunca a **coleção**. Assim esta célula pode ser executada quantas vezes você quiser sem duplicar dados — e o schema, que é caro de recriar, continua de pé.

In [ ]:
existentes = set(legislacao.tenants.get().keys())
print(f"Tenants que já existem no cluster: {sorted(existentes) or '(nenhum)'}")

# Zera os nossos tenants, para esta célula poder ser reexecutada sem duplicar dados
antigos = [t for t in (TENANT_LEIS, TENANT_LO) if t in existentes]
if antigos:
    legislacao.tenants.remove(antigos)
    print(f"🧹 Removidos para recriar: {antigos}")

legislacao.tenants.create([Tenant(name=TENANT_LEIS), Tenant(name=TENANT_LO)])
print(f"✅ Tenants criados: {TENANT_LEIS}, {TENANT_LO}")

## 4.5 Importando os dados (batch)

Nunca insira objetos um a um numa rede. O **batch** agrupa os objetos, envia em lotes paralelos e faz a vetorização em bloco.

`with ... as batch:` garante que, ao sair do bloco, tudo pendente seja enviado e a conexão fechada corretamente.

`.with_tenant(...)` é o que aponta a escrita para o tenant certo — **sem isso o Weaviate recusa a operação** numa coleção multi-tenant.

In [ ]:
import time


def importar(tenant: str, objetos: list[dict]) -> None:
    """Grava uma lista de objetos dentro de um tenant, usando batch."""
    colecao = legislacao.with_tenant(tenant)
    inicio = time.time()

    with colecao.batch.fixed_size(batch_size=100, concurrent_requests=4) as batch:
        for objeto in objetos:
            batch.add_object(properties=objeto)

    falhas = colecao.batch.failed_objects
    print(f"📦 '{tenant}': {len(colecao)} objetos em {time.time() - inicio:.1f}s · falhas: {len(falhas)}")
    for falha in falhas[:3]:
        print("   ⚠️", falha.message)


importar(TENANT_LEIS, objetos_leis)
importar(TENANT_LO, objetos_lo)

Pronto: milhares de chunks vetorizados em segundos. Agora vem a parte divertida.

## 4.6 Busca semântica (`near_text`)

A pergunta vira vetor e o Weaviate devolve os objetos mais próximos.

**Teste decisivo:** vamos procurar "motoboy que entrega comida" — palavras que **não aparecem** em nenhuma lei.

In [ ]:
from weaviate.classes.query import MetadataQuery, Filter

tenant_leis = legislacao.with_tenant(TENANT_LEIS)

resultado = tenant_leis.query.near_text(
    query="motoboy que entrega comida pela cidade",
    limit=3,
    return_metadata=MetadataQuery(distance=True),
)

for objeto in resultado.objects:
    p = objeto.properties
    print(f"[distância {objeto.metadata.distance:.3f}] {p['referencia']}")
    print(f"   {p['ementa'][:100]}\n")

Achou a lei sobre **trabalhadores de aplicativos de entrega** sem que a palavra "motoboy" existisse em lugar nenhum. É isso que a busca por palavra-chave nunca conseguiria.

> 📏 **distância** menor = mais parecido. É o inverso de "score".

## 4.7 Três buscas, lado a lado

O Weaviate oferece três estratégias. Para enxergar a diferença, precisamos de consultas que **exponham o ponto cego de cada uma**. Vamos usar duas.

In [ ]:
def comparar_buscas(consulta: str, tenant, limite: int = 3) -> None:
    """Roda a mesma consulta nas tres estrategias e imprime lado a lado."""
    print(f"🔎 CONSULTA: {consulta!r}\n" + "=" * 78)

    estrategias = {
        "BM25 (palavra-chave)": lambda: tenant.query.bm25(
            query=consulta, limit=limite, return_metadata=MetadataQuery(score=True)),
        "NEAR_TEXT (semântica)": lambda: tenant.query.near_text(
            query=consulta, limit=limite, return_metadata=MetadataQuery(distance=True)),
        "HYBRID (as duas)": lambda: tenant.query.hybrid(
            query=consulta, limit=limite, alpha=0.6, return_metadata=MetadataQuery(score=True)),
    }

    for nome, executar in estrategias.items():
        print(f"\n▸ {nome}")
        objetos = executar().objects
        if not objetos:
            print("    (nenhum resultado)")
        for objeto in objetos:
            p = objeto.properties
            rotulo = p["ementa"] or p["contexto"]
            print(f"    {p['referencia']:<22} {rotulo[:58]}")


# Caso 1: número exato de uma lei — o ponto cego da busca vetorial
comparar_buscas("7956", tenant_leis)

Resultado: o **BM25 acerta em cheio** a Lei nº 7956/2026, e a busca **vetorial erra feio**.

Faz sentido: "7956" não tem significado nenhum. O embedding de um número solto não fica perto do embedding do texto daquela lei específica. Para identificadores exatos — número de lei, CNPJ, nome próprio, código de produto — **palavra-chave ganha**.

Agora o contrário: uma pergunta em linguagem de cidadão, sem nenhuma palavra em comum com a lei.

In [ ]:
# Caso 2: linguagem natural, zero palavras em comum — o ponto cego do BM25
comparar_buscas("trabalhador que usa moto para levar pedidos", tenant_leis)

Agora inverteu: o **BM25 se perde** (vai atrás de "trabalhador" e "moto" em leis sobre outros assuntos) e a **busca vetorial encontra** a Lei nº 7.981/2026, sobre pontos de apoio para trabalhadores de aplicativos de entrega.

Repare que a consulta não tem **nenhuma** palavra em comum com a ementa: nem "aplicativo", nem "entrega", nem "motociclista". Só o significado.

### O que aprender daqui

| Estratégia | Força | Fraqueza |
|---|---|---|
| **BM25** | números, nomes próprios, códigos ("Lei 7981") | não entende sinônimos |
| **near_text** | significado, sinônimos, perguntas em linguagem natural | erra em termos exatos |
| **hybrid** | combina as duas | precisa ajustar o `alpha` |

O `alpha` é o peso: `0.0` = só BM25, `1.0` = só vetorial. **`0.6` a `0.75` costuma ser um bom ponto de partida** — e é o que vamos usar no agente.

## 4.8 Um problema que só aparece com dados reais: chunks repetidos

Olhe de novo os resultados acima: a mesma lei aparece **várias vezes**.

Não é bug nem importação duplicada — são **chunks diferentes da mesma lei**. Uma lei longa virou 11 pedaços, e vários deles falam do assunto procurado. A célula abaixo mostra o número do chunk para você confirmar com os próprios olhos.

Para o agente, porém, isso é desperdício: se os 5 resultados forem 5 pedaços da mesma lei, ele responde conhecendo **uma** lei só.

A solução é simples: pedir mais resultados do que precisa e **agrupar por documento**.

In [ ]:
def deduplicar(objetos, limite: int = 5) -> list:
    """Mantem apenas o melhor resultado de cada lei/artigo, preservando a ordem."""
    vistos, saida = set(), []
    for objeto in objetos:
        referencia = objeto.properties["referencia"]
        if referencia not in vistos:
            vistos.add(referencia)
            saida.append(objeto)
        if len(saida) == limite:
            break
    return saida


brutos = tenant_leis.query.hybrid(query="proteção e bem-estar dos animais", limit=20, alpha=0.6).objects

print(f"ANTES — {len(brutos)} chunks, repetindo a mesma lei:")
for objeto in brutos[:5]:
    p = objeto.properties
    print(f"   {p['referencia']:<20} chunk {p['chunk_id'] + 1} de {p['total_chunks']}")

print("\nDEPOIS — agrupado por lei:")
for objeto in deduplicar(brutos):
    p = objeto.properties
    print(f"   {p['referencia']:<20} {p['ementa'][:64]}")

> 🧠 Guarde o padrão: **busque com folga, agrupe depois.** Vamos usar `deduplicar` dentro das ferramentas do agente na Parte 5.

## 4.9 Filtros: combinando busca com metadados

Busca semântica **mais** condição exata. É aqui que os metadados que guardamos lá na Parte 3 pagam o investimento.

In [ ]:
resultado = tenant_leis.query.hybrid(
    query="proteção e bem-estar dos animais",
    limit=20,
    alpha=0.6,
    filters=Filter.by_property("ano").equal(2025),   # 👈 só leis de 2025
)

print("Leis de 2025 sobre animais:\n")
for objeto in deduplicar(resultado.objects):
    p = objeto.properties
    print(f"  {p['referencia']:<20} {p['ementa'][:72]}")

Filtros podem ser combinados com `&` (E) e `|` (OU):

In [ ]:
filtro = Filter.by_property("ano").greater_or_equal(2025) & Filter.by_property("chunk_id").equal(0)

resultado = tenant_leis.query.near_text(query="proteção dos animais", limit=4, filters=filtro)

print("Leis de 2025+ sobre animais (só o 1º chunk de cada):\n")
for objeto in resultado.objects:
    p = objeto.properties
    print(f"  {p['referencia']:<20} {p['ementa'][:72]}")

## 4.10 Buscando na Lei Orgânica

Mesmo código, outro tenant. Repare que aqui o campo útil é o `contexto` — a hierarquia que montamos na Parte 2.

In [ ]:
tenant_lo = legislacao.with_tenant(TENANT_LO)

resultado = tenant_lo.query.hybrid(
    query="quantos vereadores compõem a Câmara Municipal",
    limit=3, alpha=0.6,
    return_metadata=MetadataQuery(score=True),
)

for objeto in resultado.objects:
    p = objeto.properties
    print(f"[{objeto.metadata.score:.3f}] {p['referencia']} · página {p['pagina']}")
    print(f"   📂 {p['contexto'][:80]}")
    print(f"   {p['texto'][:190]}\n")

E quando o usuário cita o artigo pelo número, nem precisamos de busca — é só filtrar direto:

In [ ]:
resultado = tenant_lo.query.fetch_objects(
    filters=Filter.by_property("numero").equal("143"),
    limit=2,
)

for objeto in resultado.objects:
    p = objeto.properties
    print(f"📍 {p['referencia']} · página {p['pagina']}")
    print(f"   📂 {p['contexto']}\n")
    print(p["texto"][:400])

## 4.11 Provando o isolamento entre tenants

Esta é a célula que justifica ter escolhido multi-tenancy. Vamos fazer a **mesma pergunta**, claramente sobre a Lei Orgânica, nos **dois tenants**.

In [ ]:
pergunta = "competências privativas do Município e organização político-administrativa"

for nome, tenant in [("leis municipais", tenant_leis), ("Lei Orgânica", tenant_lo)]:
    print(f"\n▸ Buscando em: {nome}")
    for objeto in tenant.query.near_text(query=pergunta, limit=2).objects:
        p = objeto.properties
        print(f"    [{p['fonte']}] {p['referencia']:<18} {(p['ementa'] or p['contexto'])[:62]}")

O tenant das leis municipais devolve o que ele tem de mais parecido — mas **nenhum artigo da Lei Orgânica vaza para lá**. Os índices são fisicamente separados.

É exatamente essa garantia que vai permitir ao agente ter **duas ferramentas com comportamentos distintos**.

## 🎯 Exercício 2

1. Procure nas leis municipais por algo que **te interesse** (feira livre, acessibilidade, esporte, cultura...).
2. Compare `near_text` e `bm25` para a mesma consulta — qual acertou mais?
3. Descubra em que artigo da Lei Orgânica está a competência do Prefeito para vetar projetos de lei.

<details>
<summary>👀 Dica para o item 3</summary>

```python
for o in tenant_lo.query.hybrid(query="veto do Prefeito a projeto de lei", limit=3, alpha=0.6).objects:
    print(o.properties["referencia"], "·", o.properties["texto"][:160])
```
</details>

In [ ]:
# 🎯 Sua vez!

---
# Parte 5 — O Agente (30 min)

## 5.1 RAG e Agente: qual a diferença?

**RAG** (*Retrieval-Augmented Generation*) é um caminho fixo, decidido por você:

```
pergunta ──► busca ──► junta os trechos no prompt ──► LLM ──► resposta
```

Um **agente** inverte quem decide:

```
pergunta ──► LLM: "preciso buscar? onde? com quais argumentos?"
                │
                ├──► ferramenta A ──┐
                ├──► ferramenta B ──┤
                │                   ▼
                └──────────── LLM avalia o que voltou
                                    │
                        ┌───────────┴───────────┐
                        ▼                       ▼
                busca de novo            escreve a resposta
```

Você descreve **as ferramentas**; o modelo decide **quando e como** usá-las. É o que precisamos: duas bases, e uma pergunta que pode exigir uma, outra, ou as duas.

Vamos construir os dois — começando pelo RAG, para você ver o contraste.

## 5.2 Conectando o Gemini

Duas armadilhas do mundo real aqui, e as duas valem mais que o código:

**1. Nomes de modelo mudam.** Modelos antigos param de aceitar chaves novas — hoje, por exemplo, `gemini-2.5-flash` responde `404` para quem criou a chave recentemente. Em vez de fixar um nome e torcer, testamos uma lista e ficamos com o primeiro que responder.

**2. A cota grátis varia MUITO entre modelos.** No plano gratuito do Gemini:

| Modelo | Requisições/dia (free tier) |
|---|---|
| `gemini-3.6-flash` (mais capaz) | ~**20** |
| `gemini-3.5-flash-lite` | bem mais generosa |

Cada pergunta ao agente consome **2 ou mais** requisições (uma para decidir a ferramenta, outra para redigir a resposta). Com 20/dia, a aula acabaria antes da Parte 5 terminar.

Por isso a lista abaixo começa pelos modelos **`-lite`**: nesta tarefa eles roteiam tão bem quanto os grandes, e a cota aguenta a aula inteira.

In [ ]:
import warnings
from langchain_google_genai import ChatGoogleGenerativeAI

# Alguns modelos "-lite" ignoram `temperature` e avisam em toda chamada. É esperado.
warnings.filterwarnings("ignore", message=".*fixed sampling defaults.*")

# Ordem importa: os "-lite" vêm primeiro por causa da cota do plano gratuito.
CANDIDATOS = [
    "gemini-3.5-flash-lite",    # 👈 cota generosa, ótimo em tool calling
    "gemini-3.1-flash-lite",
    "gemini-3.6-flash",         # mais capaz, mas ~20 requisições/dia no free tier
    "gemini-3.5-flash",
    "gemini-2.5-flash",         # legado: só funciona com chaves antigas
]


def escolher_modelo(candidatos: list[str]) -> ChatGoogleGenerativeAI:
    """Devolve o primeiro modelo da lista que responder com a sua chave."""
    for nome in candidatos:
        try:
            llm = ChatGoogleGenerativeAI(
                model=nome,
                temperature=0,
                google_api_key=GEMINI_API_KEY,
                max_retries=3,          # tenta de novo em erro passageiro (429/503)
            )
            llm.invoke("responda apenas: ok")
            print(f"✅ Usando o modelo: {nome}")
            return llm
        except Exception as erro:
            motivo = "cota esgotada" if "RESOURCE_EXHAUSTED" in str(erro) else type(erro).__name__
            print(f"   ⏭️  {nome} indisponível ({motivo})")
    raise RuntimeError(
        "Nenhum modelo Gemini disponível. Confira a GEMINI_API_KEY "
        "ou aguarde a cota diária renovar."
    )


llm = escolher_modelo(CANDIDATOS)

> 💡 `temperature=0` deixa o modelo **determinístico** — sempre a resposta mais provável. Para consulta a legislação é o que queremos: criatividade aqui é sinônimo de invenção.
>
> Alguns modelos `-lite` usam amostragem fixa e **ignoram** esse parâmetro (por isso o `filterwarnings` acima). Na prática eles já se comportam de forma bastante estável nesta tarefa.
>
> 🚦 **Se aparecer `RESOURCE_EXHAUSTED` (429)** em qualquer célula daqui para frente, você esgotou a cota gratuita do modelo. Espere alguns minutos, ou volte a esta célula e mova outro nome para o topo de `CANDIDATOS`.

As respostas do Gemini podem vir em **blocos** (o raciocínio interno separado do texto final). Esta função extrai só o texto:

In [ ]:
def texto_da_resposta(mensagem) -> str:
    """Extrai o texto de uma mensagem do LangChain, em qualquer formato."""
    conteudo = mensagem.content
    if isinstance(conteudo, str):
        return conteudo
    partes = [b.get("text", "") for b in conteudo
              if isinstance(b, dict) and b.get("type") == "text"]
    return "\n".join(partes).strip()


print(texto_da_resposta(llm.invoke("Diga 'olá, turma!' e nada mais.")))

## 5.3 Primeiro: um RAG simples (para ter comparação)

Aqui **nós** decidimos tudo: buscar, montar o prompt, chamar o modelo.

In [ ]:
def rag_simples(pergunta: str) -> str:
    """RAG classico: busca fixa nas leis municipais e manda tudo para o LLM."""
    # 1. BUSCA
    objetos = deduplicar(tenant_leis.query.hybrid(query=pergunta, limit=20, alpha=0.6).objects)

    # 2. MONTA O CONTEXTO
    contexto = "\n\n---\n\n".join(
        f"[{o.properties['referencia']}] {o.properties['ementa']}\n{o.properties['texto']}"
        for o in objetos
    )

    # 3. PERGUNTA AO MODELO
    prompt = f"""Responda a pergunta usando APENAS os trechos de lei abaixo.
Cite sempre o número da lei. Se a resposta não estiver nos trechos, diga que não encontrou.

TRECHOS:
{contexto}

PERGUNTA: {pergunta}"""

    return texto_da_resposta(llm.invoke(prompt))


print(rag_simples("Existe alguma lei sobre pontos de apoio para entregadores?"))

Funcionou bem. Agora o problema:

In [ ]:
print(rag_simples("Quantos vereadores tem a Câmara Municipal?"))

> 🔴 **Aí está a limitação.**
>
> A resposta está na **Lei Orgânica**, mas o `rag_simples` só sabe procurar nas **leis municipais** — porque *nós* fixamos isso no código. Ele não tem como mudar de ideia.
>
> Para resolver, teríamos que escrever um classificador de perguntas na mão. **Ou** dar as duas bases ao modelo e deixá-lo escolher. É o que vamos fazer.

## 5.4 Definindo as ferramentas

Uma **ferramenta** (*tool*) é uma função Python que o modelo pode chamar. O decorador `@tool` do LangChain expõe a função ao modelo.

⚠️ **A parte mais importante desta aula inteira:**

> O modelo escolhe a ferramenta lendo a **docstring** e os **nomes dos parâmetros**. Ele não vê o corpo da função.
>
> **A docstring não é comentário — é a interface com o modelo.** Escreva-a pensando em quem vai decidir com base nela.

Repare como cada docstring abaixo diz explicitamente **quando usar** aquela ferramenta e dá exemplos.

In [ ]:
from langchain_core.tools import tool


@tool
def buscar_leis_municipais(consulta: str, ano: int | None = None) -> str:
    """Busca nas LEIS MUNICIPAIS ORDINÁRIAS de Teófilo Otoni (acervo de 2024 a 2026).

    Use para perguntas sobre regras específicas e concretas criadas pela Câmara:
    programas municipais, denominação de ruas e praças, obrigações de
    estabelecimentos comerciais, datas comemorativas, subvenções a entidades,
    políticas setoriais, utilidade pública.

    Exemplos: "existe lei sobre coleta seletiva?", "qual lei criou o programa X?"

    Args:
        consulta: o assunto procurado, em linguagem natural.
        ano: opcional. Restringe a um ano específico (ex: 2025).
    """
    # busca com folga e agrupa por lei, para não entregar 5 pedaços do mesmo documento
    objetos = tenant_leis.query.hybrid(
        query=consulta, limit=20, alpha=0.6,
        filters=Filter.by_property("ano").equal(ano) if ano else None,
        return_metadata=MetadataQuery(score=True),
    ).objects

    return json.dumps([{
        "lei": o.properties["referencia"],
        "ementa": o.properties["ementa"],
        "trecho": o.properties["texto"],
        "link": o.properties["url_pagina"],
    } for o in deduplicar(objetos)], ensure_ascii=False)


@tool
def consultar_lei_organica(consulta: str, artigo: int | None = None) -> str:
    """Consulta a LEI ORGÂNICA do Município de Teófilo Otoni — a "constituição municipal".

    Use para perguntas sobre a estrutura e as regras fundamentais do município:
    competências da Câmara e do Prefeito, número de vereadores, mandatos e posse,
    processo legislativo, vetos, orçamento público, princípios da administração,
    direitos garantidos pelo município, organização político-administrativa.

    Exemplos: "quantos vereadores tem a Câmara?", "quem pode propor emendas?"

    Args:
        consulta: o assunto procurado, em linguagem natural.
        artigo: opcional. O número exato do artigo, quando o usuário citar "Art. N".
    """
    if artigo is not None:
        objetos = tenant_lo.query.fetch_objects(
            filters=Filter.by_property("numero").equal(str(artigo)), limit=5).objects
    else:
        objetos = deduplicar(tenant_lo.query.hybrid(
            query=consulta, limit=20, alpha=0.6,
            return_metadata=MetadataQuery(score=True)).objects)

    return json.dumps([{
        "artigo": o.properties["referencia"],
        "onde_esta": o.properties["contexto"],
        "pagina": o.properties["pagina"],
        "trecho": o.properties["texto"],
    } for o in objetos], ensure_ascii=False)


print("🛠️  Ferramentas criadas:")
for ferramenta in [buscar_leis_municipais, consultar_lei_organica]:
    print(f"\n▸ {ferramenta.name}")
    print(f"  parâmetros: {list(ferramenta.args.keys())}")

Ferramentas são funções normais — dá para testar sozinhas, sem LLM nenhum:

In [ ]:
saida = consultar_lei_organica.invoke({"consulta": "número de vereadores"})
print(json.dumps(json.loads(saida)[0], indent=2, ensure_ascii=False)[:600])

## 5.5 As instruções do agente

O **system prompt** define o comportamento. Para o nosso caso, ele precisa responder a três coisas:

1. **Qual ferramenta usar quando** — a regra de roteamento
2. **Como citar** — sem fonte, a resposta não serve para nada
3. **O que fazer quando não achar** — a regra anti-invenção

In [ ]:
INSTRUCOES = """Você é um assistente especializado na legislação do município de Teófilo Otoni/MG.
Seu público são cidadãos comuns, não advogados.

Você tem DUAS bases de conhecimento separadas e precisa escolher a certa:

1. `consultar_lei_organica` — a Lei Orgânica, norma FUNDAMENTAL do município.
   Assuntos: estrutura dos poderes, competências da Câmara e do Prefeito, número de
   vereadores, mandatos, processo legislativo, vetos, orçamento, princípios gerais
   da administração pública, direitos assegurados pelo município.

2. `buscar_leis_municipais` — leis ORDINÁRIAS de 2024 a 2026.
   Assuntos: regras concretas do dia a dia, programas municipais, denominação de
   logradouros, obrigações de estabelecimentos, datas comemorativas, subvenções.

REGRAS:
1. Pergunta sobre "como o município se organiza" ou "o que a norma fundamental diz"
   → Lei Orgânica. Pergunta sobre uma regra específica e concreta → leis municipais.
2. Na dúvida, ou se a pergunta tocar os dois planos, consulte AS DUAS bases.
3. SEMPRE cite a fonte: "Art. N da Lei Orgânica" ou "Lei nº X/ANO".
4. NUNCA invente. Se as buscas não trouxerem base, diga que não encontrou e
   sugira consultar o portal da Câmara Municipal.
5. Responda em português do Brasil, de forma direta e clara para um leigo.
6. O acervo de leis ordinárias cobre apenas 2024–2026. Avise quando isso for
   relevante para a pergunta."""

print(INSTRUCOES[:300], "...")

## 5.6 Montando o agente

Com as ferramentas e as instruções prontas, o agente é **uma linha**. O `create_agent` monta por baixo o laço "pensar → chamar ferramenta → avaliar → responder".

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    llm,
    tools=[buscar_leis_municipais, consultar_lei_organica],
    system_prompt=INSTRUCOES,
)

print("🤖 Agente pronto!")

E uma função auxiliar para conversarmos com ele — mostrando **quais ferramentas** ele decidiu usar:

In [ ]:
from IPython.display import Markdown, display


def perguntar(pergunta: str, mostrar_ferramentas: bool = True):
    """Envia uma pergunta ao agente e exibe a resposta formatada."""
    display(Markdown(f"### 🙋 {pergunta}"))

    resposta = agente.invoke({"messages": [{"role": "user", "content": pergunta}]})

    if mostrar_ferramentas:
        for mensagem in resposta["messages"]:
            for chamada in getattr(mensagem, "tool_calls", None) or []:
                argumentos = ", ".join(f"{k}={v!r}" for k, v in chamada["args"].items())
                print(f"   🔧 {chamada['name']}({argumentos})")

    display(Markdown(texto_da_resposta(resposta["messages"][-1])))

## 5.7 A hora da verdade: o roteamento

Vamos fazer três perguntas de tipos diferentes. **Preste atenção na linha 🔧** — é o agente decidindo sozinho.

### Teste 1 — pergunta claramente constitucional

In [ ]:
perguntar("Quantos vereadores tem a Câmara Municipal de Teófilo Otoni?")

### Teste 2 — pergunta sobre uma regra concreta

In [ ]:
perguntar("Existe alguma lei sobre motoristas de aplicativo e entregadores na cidade?")

### Teste 3 — pergunta que atravessa as duas bases

In [ ]:
perguntar("Quem pode propor emenda à Lei Orgânica, e existe alguma lei recente "
          "sobre proteção de crianças e adolescentes no município?")

### Teste 4 — citação direta de artigo

Aqui o agente deve usar o parâmetro `artigo=` em vez de busca semântica:

In [ ]:
perguntar("O que diz o artigo 143 da Lei Orgânica?")

### Teste 5 — a pergunta que ele NÃO deve responder

Testar o que o sistema faz quando **não sabe** é tão importante quanto testar o que ele acerta.

In [ ]:
perguntar("Qual é a alíquota do IPTU em Teófilo Otoni para imóveis residenciais em 2019?")

> ✅ Um agente que admite não saber é **mais útil** que um que inventa um número plausível. A Regra 4 do system prompt e a citação obrigatória de fonte são o que produzem esse comportamento.

## 5.8 Memória: conversando de verdade

Até agora cada pergunta era independente. Para o agente entender *"e sobre esse mesmo tema?"*, ele precisa lembrar do que já foi dito.

O `checkpointer` guarda o histórico; o `thread_id` identifica a conversa (como uma sala de chat).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agente_com_memoria = create_agent(
    llm,
    tools=[buscar_leis_municipais, consultar_lei_organica],
    system_prompt=INSTRUCOES,
    checkpointer=InMemorySaver(),      # 👈 guarda o histórico
)

CONVERSA = {"configurable": {"thread_id": "aula-teofilo-otoni"}}


def conversar(pergunta: str):
    """Igual a perguntar(), mas mantendo o historico da conversa."""
    display(Markdown(f"### 🙋 {pergunta}"))
    resposta = agente_com_memoria.invoke(
        {"messages": [{"role": "user", "content": pergunta}]}, config=CONVERSA)
    for mensagem in resposta["messages"]:
        for chamada in getattr(mensagem, "tool_calls", None) or []:
            argumentos = ", ".join(f"{k}={v!r}" for k, v in chamada["args"].items())
            print(f"   🔧 {chamada['name']}({argumentos})")
    display(Markdown(texto_da_resposta(resposta["messages"][-1])))


conversar("O que a Lei Orgânica diz sobre o meio ambiente?")

In [ ]:
# Repare: não repetimos "meio ambiente". O agente entende "esse mesmo tema".
conversar("E existe alguma lei municipal recente sobre esse mesmo tema?")

## 🎯 Exercício 3

1. Faça três perguntas suas ao agente e observe o roteamento. Ele errou alguma?
2. **Quebre o agente de propósito:** troque a docstring de `buscar_leis_municipais`
   por algo vago como `"""Busca leis."""`, recrie o agente e repita o Teste 2.
   O roteamento piora? (É a melhor forma de internalizar por que a docstring importa.)
3. Acrescente uma terceira ferramenta, `listar_leis_do_ano(ano)`, que devolva as
   ementas de todas as leis de um ano. Pergunte "quantas leis foram aprovadas em 2025?"

<details>
<summary>👀 Esqueleto do item 3</summary>

```python
@tool
def listar_leis_do_ano(ano: int) -> str:
    """Lista TODAS as leis municipais aprovadas em um ano específico.

    Use quando o usuário quiser contar leis ou ver o panorama de um ano,
    em vez de procurar por um assunto.

    Args:
        ano: o ano desejado (entre 2024 e 2026).
    """
    objetos = tenant_leis.query.fetch_objects(
        filters=Filter.by_property("ano").equal(ano) & Filter.by_property("chunk_id").equal(0),
        limit=200,
    ).objects
    return json.dumps(
        [{"lei": o.properties["referencia"], "ementa": o.properties["ementa"]} for o in objetos],
        ensure_ascii=False,
    )

agente = create_agent(
    llm,
    tools=[buscar_leis_municipais, consultar_lei_organica, listar_leis_do_ano],
    system_prompt=INSTRUCOES,
)
```
</details>

In [ ]:
# 🎯 Sua vez!

---
# Parte 6 — Encerramento (5 min)

## O que você construiu

Em duas horas, saiu de uma URL pública e chegou num agente que responde sobre legislação municipal **com fonte citada**:

| Parte | O que ficou na bagagem |
|---|---|
| **1** | Python aplicado: dicionários, funções, list comprehensions, consumo de API |
| **2** | Coleta real: PDF → texto, paralelismo, regex, tolerância a falhas |
| **3** | Chunking que respeita a **estrutura** do documento, não só o tamanho |
| **4** | Busca vetorial, hybrid search, filtros e **multi-tenancy** |
| **5** | Agente com múltiplas ferramentas, roteamento por docstring e memória |

## As três ideias que valem mais que o código

1. **Metadado bom vale mais que embedding bom.** Guardar `Art. 21 · TÍTULO IV · página 16`
   junto do texto é o que transformou uma resposta plausível numa resposta verificável.

2. **A docstring é a interface com o modelo.** O agente roteia lendo a descrição das
   ferramentas. Docstring vaga = agente confuso. É código, não comentário.

3. **Separação de bases é uma decisão de arquitetura.** Multi-tenancy deu isolamento
   garantido pelo banco — não por um filtro que alguém pode esquecer de aplicar.

## Para continuar

| Ideia | Por onde começar |
|---|---|
| Ampliar o acervo | mude `ANOS` na Parte 2 e reimporte (são ~7.800 leis desde 1947) |
| Melhorar a precisão | acrescente um **reranker** ao Weaviate |
| Colocar no ar | `streamlit` ou `gradio` — poucas linhas para virar um site |
| Avaliar de verdade | monte 20 perguntas com resposta conhecida e meça o acerto |
| Outros municípios | o portal `*.mg.leg.br` é o mesmo sistema em muitas câmaras |

## Links

- 📚 [Documentação do Weaviate](https://docs.weaviate.io/) · [Weaviate Embeddings](https://docs.weaviate.io/cloud/embeddings) · [Multi-tenancy](https://docs.weaviate.io/weaviate/manage-collections/multi-tenancy)
- 🦜 [LangChain](https://docs.langchain.com/oss/python/langchain/overview) · [Ferramentas](https://docs.langchain.com/oss/python/langchain/tools) · [Agentes](https://docs.langchain.com/oss/python/langchain/agents)
- ✨ [Google AI Studio](https://aistudio.google.com/)
- 🏛️ [Câmara Municipal de Teófilo Otoni](https://www.teofilootoni.mg.leg.br/)
- 💻 [Repositório deste curso](https://github.com/dudanogueira/teofiloAI)

---

## Limpeza

Sempre feche a conexão ao terminar — o cliente do Weaviate mantém conexões abertas.

Os **dados ficam no seu cluster**, então você pode voltar a brincar com o agente depois da aula: basta rodar de novo as células 0.2, 0.4, 4.3 e a Parte 5 (pulando a importação).

Só apague os tenants se quiser mesmo zerar o cluster — a linha está comentada de propósito.

In [ ]:
# Descomente a linha abaixo APENAS se quiser apagar os dados do seu cluster:
# legislacao.tenants.remove([TENANT_LEIS, TENANT_LO])

client.close()
print("👋 Conexão encerrada. Seus dados continuam no cluster.")
print("   Obrigado e até a próxima!")